# RAG with LangChain + Ollama

Fully offline: a local LLM answers questions using retrieved text from NASA's Systems Engineering Handbook, indexed with LangChain + ChromaDB.

## 1. Connect to the local model

`ChatOllama` wraps your local Ollama server, with no hosted API key required.

In [ ]:
%pip install -q langchain langchain-community langchain-ollama langchain-chroma langchain-text-splitters chromadb pypdf


In [ ]:
from langchain_ollama import ChatOllama

Language_Model = 'phi3:mini'
Embedding_Model = 'nomic-embed-text'

llm = ChatOllama(model=Language_Model)
response = llm.invoke("Reply with a sentence to confirm that you are working")
print(response.content)

## 2. Load the manual

`PyPDFLoader` loads the PDF as one `Document` per page, tagged with page metadata.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

Corpus = "nasa_systems_engineering_handbook.pdf"
loader = PyPDFLoader(Corpus)
pages = loader.load()

print(f'Total no of pages is {len(pages)}')
print(pages[0].page_content[:50])

## 3. Chunk the text

`RecursiveCharacterTextSplitter` splits pages into overlapping text windows so facts near a boundary have a better chance of remaining in context.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(pages)

print(f'Total no of chunks is {len(chunks)}')
print("\nFirst chunk preview:\n", chunks[0].page_content[:500])
print("Metadata:", chunks[0].metadata)

## 4. Embed and store in ChromaDB

`OllamaEmbeddings` converts chunks to vectors. `Chroma.from_documents` embeds and stores them in a local persistent vector database.

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model=Embedding_Model)

vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nasa_handbook",
    persist_directory="./chroma_db"
)

print("Indexing complete")
print("Number of chunks:", len(chunks))

## 5. Retrieve relevant chunks

`similarity_search` embeds the question and returns the `k` closest chunks. This is retrieval only; the LLM has not answered yet.

In [ ]:
def retrieve(question, top_k=3):
    results = vectorStore.similarity_search(question, k=top_k)
    for doc in results:
        print(f"[page {doc.metadata.get('page_label', doc.metadata.get('page', 'unknown'))}]")
        print(doc.page_content[:300])
        print()
    return results

retrieve("What is the purpose of systems engineering?")

## 6. Generate a grounded answer

Retrieved chunks are passed to the LLM with an instruction to answer only from the supplied context. Prompt instructions help, but are not a guarantee against hallucinations.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    """Answer the question ONLY from the context below. If the context does not contain the answer, say you cannot answer from the provided manual. Do not guess.

    Context:
{context}

    Question: {question}
    """
)

def ask(question, top_k=3):
    results = retrieve(question, top_k=top_k)
    context = "\n\n".join(doc.page_content for doc in results)
    chain = prompt_template | llm
    response = chain.invoke({"context": context, "question": question})
    print(response.content)
    return response.content

ask("What is the purpose of systems engineering?")

## 7. Guard against hallucination

A distance threshold can refuse questions whose nearest retrieved chunk appears too far away. This is a heuristic, not a guarantee; tune it with covered and uncovered test questions.

In [ ]:
DISTANCE_THRESHOLD = 0.7  # starting point; calibrate using real questions

def ask_safely(question, top_k=3):
    results = vectorStore.similarity_search_with_score(question, k=top_k)
    if not results:
        print("No passages were retrieved. Not guessing.")
        return None

    best_doc, best_distance = results[0]
    print(f"Best retrieval distance: {best_distance:.3f}")

    if best_distance > DISTANCE_THRESHOLD:
        print(f"Best match distance exceeds threshold ({DISTANCE_THRESHOLD}). This may be outside the manual's coverage. Not guessing.")
        return None

    return ask(question, top_k=top_k)

## 8. Try the guardrail

Compare one covered question with one likely out-of-scope question. Inspect their distances; the threshold may need calibration for your embedding model and vector-store metric.

In [ ]:
print("=== Covered question ===")
ask_safely("What is the purpose of systems engineering?")

print("\n=== Likely out-of-scope question ===")
ask_safely("How often should pulley bearings be lubricated on a conveyor belt?")

## Notes and next steps

- Keep the PDF in the same directory as the notebook, or update `Corpus`.
- Install and pull the local models before running: `ollama pull phi3:mini` and `ollama pull nomic-embed-text`.
- The local `chroma_db/` folder is generated data and does not need to be committed.
- Test retrieval and calibrate `DISTANCE_THRESHOLD` against covered and uncovered questions; a distance threshold is not a proof that a response is hallucination-free.